# ReLU Single-Path Cell: testing the lens / positive-feedback narrative

This notebook is the ReLU analog of the **single-path DLGN** study in
*Layers as Lenses* (paper Sec. 3), following **Track A** of
`notes/research_spec.md` ("ReLU Lens Theorem Elicitation").

**Why a "cell" and not a width-1 chain.** In the single-path DLGN, the $D$ gating
layers hold one neuron each, and the lens on layer $\ell$ is the product of the
*other* layers' gates. A literal width-1 ReLU chain `[1, 1, 1]` is degenerate: it is
a single-index function with no lens at all. The minimal non-degenerate ReLU object
is the **ReLU cell**

$$\texttt{hidden\_dims} = [D, 1, 1], \qquad \text{bias}=\text{False},$$

on depth-$D$ COB-ODT data. The $D$ first-layer neurons play the role of the $D$ DLGN
layers (they are honest input-space halfspaces); the width-1 downstream chain supplies
two *deep* gates whose activation regions are carved by all $D$ first-layer halfspaces.
That downstream chain is the ReLU lens.

**The lens.** For first-layer neuron $m$ the gradient factorises as

$$\nabla_{W_1[m,:]}\mathcal{L}
= \mathbb{E}\Big[\underbrace{\sigma(-y\hat y)}_{\text{Residue}}\;
\underbrace{\Lambda_m(x)}_{\text{Lens}}\;
\underbrace{\mathbb{1}[W_1[m,:]^\top x>0]}_{\text{Blinder}}\;(-y)\,x\Big],$$

with $\Lambda_m(x)=\sum_{\pi\ni m} a_\pi \prod_{\ell\ge 2}\mathbb{1}[\cdot]$ the
path-summed downstream coefficient, computed here by
`first_layer_neuron_weighted_lens`.

**Positive feedback.** $\Lambda_m$ is built from the downstream gates, which are
themselves shaped by *all* first-layer neurons. Each neuron is therefore both a
separator (on data weighted by its own lens) and a component of everyone else's lens.
The experiments below ask whether that loop is what organises training:

| Section | Hypothesis (`notes/research_spec.md`) | What it shows |
| --- | --- | --- |
| Signed-cos trajectories | H1, H3, CT3 | Do first-layer neurons lock onto ODT directions $\pm u_i$, and in what order? |
| Convergence labels ($\upsilon$-tuple) | Track A.1, CT4 | Does the cell land on a *root-to-leaf path* solution, as the DLGN does? |
| Capture epochs | H2, H3 | Hierarchical (parent-before-child) feature emergence |
| Lens leaf-concentration | H8 | Does each neuron's lens sharpen onto one ODT leaf region? |
| Gradient probe by ODT level | H7, CT1 | Is the early gradient leaf-dominated, as at the DLGN saddle? |
| Frozen-downstream control | **H11** | Kill lens evolution: does alignment stall? (the causal test of positive feedback) |
| Width-1 first layer control | Track A.3 | No lens at all: collapse to a single-index solution |
| Seed census | Track A.1 | Is the solution set discrete, like the DLGN's $2^{D-2}D!$ family? |

All training/analysis logic comes from `src/layer_lenses/`; this notebook only
composes it.

In [ ]:
from pathlib import Path
import pickle
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

%load_ext autoreload
%autoreload 2

from layer_lenses.odt import odt_leaf_ids_for_x
from layer_lenses.relu_analysis import (
    activation_tensor_by_layer_neuron_data,
    collect_start_end_metrics,
    first_layer_neuron_weighted_lens,
    first_layer_odt_alignment,
    log_loss_gradients,
    plot_first_layer_odt_alignment,
    plot_layered_relu_graph,
    run_single_relu_seed,
)
from layer_lenses.relu_training import checkpoint_model_from_state
from layer_lenses.utilities import plot_complete_binary_tree_values

print("cuda available:", torch.cuda.is_available())

## 1. Configuration

Defaults follow the Track A recommendation in `notes/research_spec.md`: primary depth
$D=4$ (15 internal nodes, 16 leaves, the paper's cleaner setting), $d=100$,
`bias=False`, full-batch gradient descent, and **weight decay 0** so that any cleanup
of the alignment is attributable to the dynamics rather than to the regulariser.

In [ ]:
# --- data / architecture ---------------------------------------------------
DIM = 100
DEPTH = 4                           # ODT depth D
CELL_HIDDEN_DIMS = [DEPTH, 1, 1]    # the ReLU cell: D first-layer neurons + width-1 chain
N_TRAIN = 200_000
BIAS = False
ODT_HYPERPLANES = "random_orthogonal"   # or "coordinate_axes" for an easier-to-read run

# --- optimisation ----------------------------------------------------------
TOTAL_EPOCHS = 4_000
LR = 2e-3
BATCH_SIZE = N_TRAIN                # full-batch GD, as in the single-path DLGN study
WEIGHT_DECAY = 0.0
OPTIMIZER = "adamw"                 # "adamw" or "sgd"
MOMENTUM = 0.0                      # used only by "sgd"

# Dense snapshots early (the lens reorients there), sparser later.
SNAPSHOT_EPOCHS = tuple(sorted(set(range(0, 200)) | set(range(0, TOTAL_EPOCHS + 1, 25))))

# --- analysis thresholds ---------------------------------------------------
COS_THRESHOLD = 0.85                # |cos| above which a neuron counts as ODT-aligned
NORM_THRESHOLD = 0.0                # optional extra norm gate for "capture"

SEED = 5178
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = Path("../results/cache")

NUM_INTERNAL_NODES = 2**DEPTH - 1
NUM_LEAVES = 2**DEPTH
NODE_LEVEL = np.floor(np.log2(np.arange(NUM_INTERNAL_NODES) + 1)).astype(int)

print(f"ReLU cell {CELL_HIDDEN_DIMS} on depth-{DEPTH} ODT "
      f"({NUM_INTERNAL_NODES} internal nodes, {NUM_LEAVES} leaves), d={DIM}")
print(f"{len(SNAPSHOT_EPOCHS)} snapshot epochs, device={DEVICE}")

### Run wrapper and small helpers

One wrapper keeps every run in this notebook on the same data/optimiser settings, so
the controls differ from the reference run only in the thing being controlled.

In [ ]:
def run_relu_cell(
    seed: int,
    *,
    hidden_dims: list[int] | None = None,
    total_epochs: int | None = None,
    snapshot_epochs=None,
    parameter_update_mask: dict[str, np.ndarray] | None = None,
    show_progress: bool = True,
) -> dict[str, Any]:
    """Train one ReLU cell on COB-ODT data using the notebook-level defaults."""
    hidden_dims = list(CELL_HIDDEN_DIMS if hidden_dims is None else hidden_dims)
    total_epochs = TOTAL_EPOCHS if total_epochs is None else total_epochs
    if snapshot_epochs is None:
        snapshot_epochs = tuple(e for e in SNAPSHOT_EPOCHS if e <= total_epochs)
    return run_single_relu_seed(
        seed,
        dim=DIM,
        depth=DEPTH,
        n_train=N_TRAIN,
        hidden_dims=hidden_dims,
        total_epochs=total_epochs,
        lr=LR,
        batch_size=BATCH_SIZE,
        weight_decay=WEIGHT_DECAY,
        snapshot_epochs=snapshot_epochs,
        bias=BIAS,
        odt_hyperplanes=ODT_HYPERPLANES,
        optimizer=OPTIMIZER,
        momentum=MOMENTUM,
        parameter_update_mask=parameter_update_mask,
        device=DEVICE,
        show_progress=show_progress,
    )


def save_seed_result_pickle(seed_result: dict[str, Any], path: str | Path) -> Path:
    """Save a seed_result dictionary to a pickle file."""
    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("wb") as f:
        pickle.dump(seed_result, f, protocol=pickle.HIGHEST_PROTOCOL)
    return out_path


def load_seed_result_pickle(path: str | Path) -> dict[str, Any]:
    """Load a seed_result dictionary from a pickle file."""
    with Path(path).open("rb") as f:
        return pickle.load(f)


def node_label(node_id: int) -> str:
    """Readable name for an ODT internal node, e.g. ``u3 (L2)``."""
    return f"u{node_id} (L{int(np.floor(np.log2(node_id + 1)))})"


def ancestors_of_leaf(leaf_node_id: int) -> list[int]:
    """Internal nodes on the root-to-leaf path of a heap-indexed leaf, root first."""
    path: list[int] = []
    node = leaf_node_id
    while node > 0:
        node = (node - 1) // 2
        path.append(node)
    return path[::-1]


ROOT_TO_LEAF_PATHS = {
    leaf: ancestors_of_leaf(leaf)
    for leaf in range(NUM_INTERNAL_NODES, NUM_INTERNAL_NODES + NUM_LEAVES)
}


def matching_root_to_leaf_path(node_ids) -> int | None:
    """Return a leaf id whose root-to-leaf path equals ``node_ids`` as a set, else None."""
    wanted = set(int(n) for n in node_ids)
    for leaf, path in ROOT_TO_LEAF_PATHS.items():
        if set(path) == wanted:
            return leaf
    return None

## 2. Reference run

The expensive cell. One ReLU cell, one seed, snapshot-rich.

In [ ]:
print(f"Running ReLU cell {CELL_HIDDEN_DIMS}, seed {SEED} ...")
seed_result = run_relu_cell(SEED)

tree = seed_result["tree"]
ckpts = seed_result["out"]["checkpoint_snapshots"]
x_train = seed_result["data"]["x_train"]
y_train = seed_result["data"]["y_train"]
epochs_available = sorted(ckpts)
FINAL_EPOCH = epochs_available[-1]

print("done. checkpoints:", len(epochs_available), "final epoch:", FINAL_EPOCH)
pd.DataFrame(collect_start_end_metrics(seed_result))

In [ ]:
CACHE_PATH = CACHE_DIR / (
    f"relu_cell_seed{SEED}_d{DIM}_D{DEPTH}_"
    f"{'x'.join(str(w) for w in CELL_HIDDEN_DIMS)}_{OPTIMIZER}.pkl"
)

# save_seed_result_pickle(seed_result, CACHE_PATH)

# seed_result = load_seed_result_pickle(CACHE_PATH)
# tree = seed_result["tree"]
# ckpts = seed_result["out"]["checkpoint_snapshots"]
# x_train = seed_result["data"]["x_train"]
# y_train = seed_result["data"]["y_train"]
# epochs_available = sorted(ckpts)
# FINAL_EPOCH = epochs_available[-1]
print("cache path:", CACHE_PATH)

### Loss curve and dead-path check

A `[D, 1, 1]` cell can fail in a specific way: if a width-1 downstream neuron is
inactive on (almost) all data, the lens is identically zero and no first-layer neuron
receives gradient — the ReLU analog of a dead run. Active fractions near 0 or 1 for
hidden layers 2-3 flag such a run, which should be analysed separately rather than
dropped silently.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
losses = seed_result["out"]["epoch_losses"]
axes[0].plot(losses)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("train log loss")
axes[0].set_title(f"Seed {SEED}: training loss")
axes[0].axhline(np.log(2), ls="--", c="gray", lw=1, label="log 2 (chance)")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(losses)
axes[1].set_xscale("log")
axes[1].set_xlabel("epoch (log)")
axes[1].set_ylabel("train log loss")
axes[1].set_title("early dynamics")
axes[1].grid(alpha=0.2)
fig.tight_layout()
plt.show()


final_model = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[FINAL_EPOCH])
final_acts = activation_tensor_by_layer_neuron_data(
    model=final_model, x_eval=x_train, device=DEVICE
)

activity_rows = []
for layer_idx, width in enumerate(CELL_HIDDEN_DIMS):
    for neuron_id in range(width):
        activity_rows.append(
            {
                "layer": layer_idx,
                "neuron_id": neuron_id,
                "active_fraction": float(final_acts[layer_idx, neuron_id].mean()),
            }
        )
activity_df = pd.DataFrame(activity_rows)
activity_df

## 3. Signed-cosine trajectories (H1, H3, CT3)

The direct analog of the paper's single-path parameter-trajectory figure: for every
first-layer neuron, its signed cosine with every ODT normal $u_i$ over training,
alongside its norm. The DLGN prediction is a *staged* picture — an early symmetric
phase near the saddle, then separation, with different neurons locking onto different
nodes along one root-to-leaf path.

Signed (not absolute) cosine matters here: ReLU has no sign symmetry, so $+u_i$ and
$-u_i$ are genuinely different solutions.

In [ ]:
def signed_cos_trajectories(ckpts: dict[int, dict[str, torch.Tensor]], tree):
    """Return (epochs, signed_cos[E, M, N_nodes], norms[E, M]) over all checkpoints."""
    epochs = np.array(sorted(ckpts), dtype=int)
    cos_list, norm_list = [], []
    for epoch in epochs:
        alignment = first_layer_odt_alignment(state_dict=ckpts[int(epoch)], tree=tree)
        cos_list.append(alignment["cos"].numpy())
        norm_list.append(alignment["row_norms"].numpy())
    return epochs, np.stack(cos_list), np.stack(norm_list)


traj_epochs, traj_cos, traj_norms = signed_cos_trajectories(ckpts, tree)
print("signed cos trajectory:", traj_cos.shape, "(epochs, first-layer neurons, ODT nodes)")

In [ ]:
TRAJ_TOP_K = 3           # ODT nodes to highlight per neuron (by final |cos|)
TRAJ_EPOCH_LIMIT = None  # e.g. 400 to zoom into the early phase

mask = slice(None) if TRAJ_EPOCH_LIMIT is None else traj_epochs <= TRAJ_EPOCH_LIMIT
ep = traj_epochs[mask]
num_neurons = traj_cos.shape[1]

fig, axes = plt.subplots(
    2, num_neurons, figsize=(4.2 * num_neurons, 7), sharex=True, squeeze=False
)
level_colors = plt.get_cmap("viridis", DEPTH)

for neuron_id in range(num_neurons):
    ax = axes[0, neuron_id]
    final_abs = np.abs(traj_cos[-1, neuron_id])
    top_nodes = np.argsort(final_abs)[::-1][:TRAJ_TOP_K]
    for node_id in range(NUM_INTERNAL_NODES):
        if node_id in top_nodes:
            continue
        ax.plot(ep, traj_cos[mask, neuron_id, node_id], color="0.85", lw=0.8, zorder=1)
    for node_id in top_nodes:
        ax.plot(
            ep,
            traj_cos[mask, neuron_id, node_id],
            lw=1.8,
            color=level_colors(NODE_LEVEL[node_id]),
            label=node_label(node_id),
            zorder=2,
        )
    ax.axhline(0.0, color="k", lw=0.6)
    for sign in (1, -1):
        ax.axhline(sign * COS_THRESHOLD, color="crimson", ls=":", lw=0.9)
    ax.set_ylim(-1.05, 1.05)
    ax.set_title(f"first-layer neuron {neuron_id}")
    ax.legend(fontsize=7, loc="lower left")
    ax.grid(alpha=0.2)
    if neuron_id == 0:
        ax.set_ylabel(r"signed $\cos(w_m, u_i)$")

    ax_norm = axes[1, neuron_id]
    ax_norm.plot(ep, traj_norms[mask, neuron_id], color="tab:blue")
    ax_norm.set_xlabel("epoch")
    ax_norm.grid(alpha=0.2)
    if neuron_id == 0:
        ax_norm.set_ylabel(r"$\|w_m\|$")

fig.suptitle(
    f"Seed {SEED}: first-layer signed cosine to ODT normals (colour = ODT level)",
    y=1.02,
)
fig.tight_layout()
plt.show()

## 4. Convergence labels: the $\upsilon$-tuple and the path test (Track A.1, CT4)

Label each first-layer neuron by its signed argmax-cosine ODT node if
$|\cos|\ge\tau$, else "unaligned". The tuple of labels is the run's solution
identity, the ReLU analog of the DLGN's $2^{D-2}D!$ solution set.

**The test that matters:** in the single-path DLGN each layer locks onto a distinct
node along *one root-to-leaf path*. If the ReLU cell's $D$ aligned first-layer neurons
also form a root-to-leaf path, the generalisation holds at cell scale.

In [ ]:
def neuron_labels(
    state_dict: dict[str, torch.Tensor],
    tree,
    *,
    cos_threshold: float = COS_THRESHOLD,
    norm_threshold: float = NORM_THRESHOLD,
) -> pd.DataFrame:
    """Label each first-layer neuron by its signed argmax-cosine ODT node."""
    alignment = first_layer_odt_alignment(state_dict=state_dict, tree=tree)
    cos = alignment["cos"].numpy()
    norms = alignment["row_norms"].numpy()
    best_node = np.abs(cos).argmax(axis=1)
    rows = []
    for neuron_id, node_id in enumerate(best_node):
        signed_cos = float(cos[neuron_id, node_id])
        aligned = abs(signed_cos) >= cos_threshold and norms[neuron_id] >= norm_threshold
        rows.append(
            {
                "neuron_id": neuron_id,
                "norm": float(norms[neuron_id]),
                "odt_node": int(node_id),
                "odt_level": int(NODE_LEVEL[node_id]),
                "signed_cos": signed_cos,
                "aligned": bool(aligned),
                "label": (
                    f"{'+' if signed_cos > 0 else '-'}u{int(node_id)}"
                    if aligned
                    else "unaligned"
                ),
            }
        )
    return pd.DataFrame(rows)


def upsilon_tuple(labels_df: pd.DataFrame) -> tuple[str, ...]:
    """Solution identity of a run: the ordered tuple of first-layer neuron labels."""
    return tuple(labels_df.sort_values("neuron_id")["label"])


final_labels = neuron_labels(ckpts[FINAL_EPOCH], tree)
captured_nodes = sorted(final_labels.loc[final_labels["aligned"], "odt_node"])
matched_leaf = matching_root_to_leaf_path(captured_nodes)

print("upsilon-tuple:", upsilon_tuple(final_labels))
print("distinct captured nodes:", captured_nodes)
print(
    "root-to-leaf path solution:",
    f"yes (leaf {matched_leaf}, path {ROOT_TO_LEAF_PATHS[matched_leaf]})"
    if matched_leaf is not None
    else "no",
)
final_labels

## 5. Capture epochs and hierarchical emergence (H2, H3)

Capture epoch of ODT node $u_i$ = first checkpoint at which some first-layer neuron
exceeds the cosine (and norm) threshold on it. The DLGN narrative predicts a
hierarchical ordering visible on the tree plot: parents before their children.

In [ ]:
def first_capture_epochs(
    epochs: np.ndarray,
    signed_cos: np.ndarray,
    norms: np.ndarray,
    *,
    cos_threshold: float = COS_THRESHOLD,
    norm_threshold: float = NORM_THRESHOLD,
) -> pd.DataFrame:
    """First epoch at which each ODT node is captured by some first-layer neuron."""
    hit = (np.abs(signed_cos) >= cos_threshold) & (
        norms[:, :, None] >= norm_threshold
    )
    rows = []
    for node_id in range(signed_cos.shape[2]):
        epoch_idx, neuron_idx = np.nonzero(hit[:, :, node_id])
        if epoch_idx.size == 0:
            rows.append(
                {
                    "odt_node_id": node_id,
                    "odt_level": int(NODE_LEVEL[node_id]),
                    "first_epoch": np.nan,
                    "neuron_id": np.nan,
                    "signed_cos": np.nan,
                }
            )
            continue
        first = int(epoch_idx.min())
        neuron = int(neuron_idx[epoch_idx == first][0])
        rows.append(
            {
                "odt_node_id": node_id,
                "odt_level": int(NODE_LEVEL[node_id]),
                "first_epoch": float(epochs[first]),
                "neuron_id": neuron,
                "signed_cos": float(signed_cos[first, neuron, node_id]),
            }
        )
    return pd.DataFrame(rows)


capture_df = first_capture_epochs(traj_epochs, traj_cos, traj_norms)
capture_vec = (
    capture_df.set_index("odt_node_id")["first_epoch"]
    .reindex(range(NUM_INTERNAL_NODES))
    .to_numpy(dtype=float)
)

fig_tree, ax_tree = plot_complete_binary_tree_values(
    capture_vec,
    title=f"Seed {SEED}: first capture epoch per ODT node (|cos| > {COS_THRESHOLD})",
    suptitle="",
)
plt.show()
capture_df

## 6. Norm vs alignment scatter

The standard view from the multi-path runs, kept here for comparability: with only $D$
first-layer neurons every point is individually interpretable.

In [ ]:
SCATTER_EPOCH = FINAL_EPOCH

alignment, fig_scatter, ax_scatter = plot_first_layer_odt_alignment(
    state_dict=ckpts[SCATTER_EPOCH],
    tree=tree,
    depth=DEPTH,
    epoch=SCATTER_EPOCH,
    seed=SEED,
)
plt.show()

## 7. The lens itself (H8)

$\Lambda_m(x)$ is the summed weight of all active paths from first-layer neuron $m$ to
the output — exactly what `first_layer_neuron_weighted_lens` returns, one value per
training point. Two things to look at:

1. **Leaf concentration.** Bin $|\Lambda_m|$ by the ODT leaf each point falls into and
   take the entropy of the resulting distribution. A diffuse lens has entropy near
   $\log(2^D)$; a lens that has specialised onto one leaf region drops well below it.
2. **Sharpening over training.** The same entropy tracked across checkpoints. Falling
   entropy is the lens half of the feedback loop tightening.

In [ ]:
leaf_ids = odt_leaf_ids_for_x(x_train, tree)
leaf_offsets = leaf_ids - NUM_INTERNAL_NODES


def first_layer_lens_matrix(
    *,
    model,
    x: np.ndarray,
    node_mask=None,
    device: str = DEVICE,
    activation_tensor: np.ndarray | None = None,
) -> np.ndarray:
    """Lens value per (first-layer neuron, data point): shape (width_0, n)."""
    if activation_tensor is None:
        activation_tensor = activation_tensor_by_layer_neuron_data(
            model=model, x_eval=x, device=device
        )
    state_dict = model.state_dict()
    width0 = model.hidden_layers[0].out_features
    return np.stack(
        [
            first_layer_neuron_weighted_lens(
                state_dict=state_dict,
                activation_tensor=activation_tensor,
                first_layer_neuron_id=neuron_id,
                node_mask=node_mask,
            )
            for neuron_id in range(width0)
        ]
    )


def lens_leaf_profile(lens: np.ndarray, leaf_offsets: np.ndarray) -> pd.DataFrame:
    """Per-neuron leaf concentration of |lens|: mass, entropy, dominant leaf."""
    rows = []
    for neuron_id, values in enumerate(lens):
        mass = np.bincount(
            leaf_offsets, weights=np.abs(values), minlength=NUM_LEAVES
        ).astype(float)
        total = mass.sum()
        if total <= 0:
            rows.append(
                {
                    "neuron_id": neuron_id,
                    "lens_active_fraction": float((values != 0).mean()),
                    "mean_abs_lens": 0.0,
                    "leaf_entropy_nats": np.nan,
                    "dominant_leaf_offset": -1,
                    "dominant_leaf_share": np.nan,
                }
            )
            continue
        p = mass / total
        nonzero = p[p > 0]
        rows.append(
            {
                "neuron_id": neuron_id,
                "lens_active_fraction": float((values != 0).mean()),
                "mean_abs_lens": float(np.abs(values).mean()),
                "leaf_entropy_nats": float(-(nonzero * np.log(nonzero)).sum()),
                "dominant_leaf_offset": int(p.argmax()),
                "dominant_leaf_share": float(p.max()),
            }
        )
    return pd.DataFrame(rows)


final_lens = first_layer_lens_matrix(
    model=final_model, x=x_train, activation_tensor=final_acts
)
final_lens_profile = lens_leaf_profile(final_lens, leaf_offsets)
print(f"uniform-over-leaves entropy = log({NUM_LEAVES}) = {np.log(NUM_LEAVES):.3f} nats")
final_lens_profile

In [ ]:
LENS_TRACK_EPOCHS = [
    epochs_available[i]
    for i in np.unique(np.linspace(0, len(epochs_available) - 1, 12).astype(int))
]

lens_entropy_rows = []
for epoch in LENS_TRACK_EPOCHS:
    model_at_epoch = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[epoch])
    lens = first_layer_lens_matrix(model=model_at_epoch, x=x_train)
    profile = lens_leaf_profile(lens, leaf_offsets)
    profile["epoch"] = epoch
    lens_entropy_rows.append(profile)

lens_entropy_df = pd.concat(lens_entropy_rows, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for neuron_id, group in lens_entropy_df.groupby("neuron_id"):
    axes[0].plot(group["epoch"], group["leaf_entropy_nats"], marker="o", ms=3,
                 label=f"neuron {neuron_id}")
    axes[1].plot(group["epoch"], group["lens_active_fraction"], marker="o", ms=3,
                 label=f"neuron {neuron_id}")
axes[0].axhline(np.log(NUM_LEAVES), ls="--", c="gray", lw=1, label="uniform over leaves")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("entropy of |lens| over ODT leaves (nats)")
axes[0].set_title("H8: does each neuron's lens concentrate?")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("fraction of points with non-zero lens")
axes[1].set_title("lens support")
axes[1].grid(alpha=0.2)
fig.tight_layout()
plt.show()

lens_entropy_df.pivot(index="epoch", columns="neuron_id", values="leaf_entropy_nats")

## 8. Gradient probe by ODT level (H7, CT1)

CT1 predicts that from a generic (near-orthogonal to ODT) initialisation, the expected
first-layer gradient has its mass along **leaf** directions, weighted by path
coefficients — the ReLU version of the DLGN saddle $W^*=c\sum_j -u_j$. Later in
training, as the lens concentrates, mass should move to the internal nodes on the path
each neuron is claiming.

This uses exact full-data gradients from `log_loss_gradients` (no weight updates).

In [ ]:
def first_layer_grad_odt_components(
    *,
    model,
    x: np.ndarray,
    y: np.ndarray,
    tree,
    device: str = DEVICE,
) -> np.ndarray:
    """Components of the first-layer weight gradient along each ODT normal: (M, N)."""
    grads = log_loss_gradients(model=model, x=x, y=y, device=device)
    grad_w1 = grads["hidden_layers.0.weight"].numpy()
    odt_wts = tree.w_list / np.linalg.norm(tree.w_list, axis=1, keepdims=True)
    return grad_w1 @ odt_wts.T


GRAD_PROBE_EPOCHS = [
    epochs_available[i]
    for i in np.unique(np.linspace(0, len(epochs_available) - 1, 6).astype(int))
]

grad_rows = []
for epoch in GRAD_PROBE_EPOCHS:
    model_at_epoch = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[epoch])
    components = first_layer_grad_odt_components(
        model=model_at_epoch, x=x_train, y=y_train, tree=tree
    )
    for neuron_id in range(components.shape[0]):
        for node_id in range(NUM_INTERNAL_NODES):
            grad_rows.append(
                {
                    "epoch": epoch,
                    "neuron_id": neuron_id,
                    "odt_node_id": node_id,
                    "odt_level": int(NODE_LEVEL[node_id]),
                    "grad_component": float(components[neuron_id, node_id]),
                }
            )
grad_df = pd.DataFrame(grad_rows)
grad_df["abs_component"] = grad_df["grad_component"].abs()

level_share = (
    grad_df.groupby(["epoch", "odt_level"])["abs_component"].sum().unstack("odt_level")
)
level_share = level_share.div(level_share.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(7, 4.5))
for level in level_share.columns:
    ax.plot(level_share.index, level_share[level], marker="o", ms=4,
            color=plt.get_cmap("viridis", DEPTH)(level), label=f"ODT level {level}")
ax.set_xlabel("epoch")
ax.set_ylabel("share of total |gradient . u| mass")
ax.set_title("CT1: which ODT levels does the first-layer gradient point along?")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

level_share

## 9. Cell as a graph

The whole cell fits in one picture: first-layer neurons annotated with their ODT
labels, feeding the width-1 lens chain.

In [ ]:
GRAPH_EPOCH = FINAL_EPOCH
NODE_MASK = [np.ones(width, dtype=bool) for width in CELL_HIDDEN_DIMS]

graph_alignment, visible_edges, fig_graph, ax_graph = plot_layered_relu_graph(
    state_dict=ckpts[GRAPH_EPOCH],
    tree=tree,
    depth=DEPTH,
    epoch=GRAPH_EPOCH,
    node_mask=NODE_MASK,
    alignment_abs_threshold=COS_THRESHOLD,
    edge_abs_threshold=0.0,
    figsize=(9, 7),
)
plt.show()

## 10. Control A — frozen downstream: the positive-feedback ablation (H11)

The causal test. Freeze hidden layers 2-3 and the output layer at initialisation and
train only the first layer. The lens is then static: neurons still fit lens-weighted
data, but they no longer shape each other's lens, so the feedback loop is cut. If the
narrative is right, ODT alignment should stall, lose its hierarchical ordering, or
capture strictly fewer nodes than the free run.

Same seed, so data and initialisation are identical to the reference run.

In [ ]:
def freeze_downstream_mask(hidden_dims: list[int], dim: int) -> dict[str, np.ndarray]:
    """Update mask that trains the first hidden layer only (bias=False models)."""
    mask = {"hidden_layers.0.weight": np.ones((hidden_dims[0], dim), dtype=np.float32)}
    prev = hidden_dims[0]
    for layer_idx, width in enumerate(hidden_dims[1:], start=1):
        mask[f"hidden_layers.{layer_idx}.weight"] = np.zeros((width, prev), dtype=np.float32)
        prev = width
    mask["output_layer.weight"] = np.zeros((1, prev), dtype=np.float32)
    return mask


frozen_result = run_relu_cell(
    SEED,
    parameter_update_mask=freeze_downstream_mask(CELL_HIDDEN_DIMS, DIM),
)
frozen_ckpts = frozen_result["out"]["checkpoint_snapshots"]
frozen_final_epoch = max(frozen_ckpts)
frozen_labels = neuron_labels(frozen_ckpts[frozen_final_epoch], frozen_result["tree"])

frozen_epochs, frozen_cos, frozen_norms = signed_cos_trajectories(
    frozen_ckpts, frozen_result["tree"]
)
frozen_capture = first_capture_epochs(frozen_epochs, frozen_cos, frozen_norms)

comparison = pd.DataFrame(
    [
        {
            "run": "free (lens evolves)",
            "final_train_log_loss": collect_start_end_metrics(seed_result)[-1][
                "train_log_loss"
            ],
            "final_test_error": collect_start_end_metrics(seed_result)[-1]["test_error"],
            "aligned_neurons": int(final_labels["aligned"].sum()),
            "captured_nodes": int(capture_df["first_epoch"].notna().sum()),
            "max_abs_cos": float(final_labels["signed_cos"].abs().max()),
            "upsilon": str(upsilon_tuple(final_labels)),
        },
        {
            "run": "frozen downstream (no lens evolution)",
            "final_train_log_loss": collect_start_end_metrics(frozen_result)[-1][
                "train_log_loss"
            ],
            "final_test_error": collect_start_end_metrics(frozen_result)[-1]["test_error"],
            "aligned_neurons": int(frozen_labels["aligned"].sum()),
            "captured_nodes": int(frozen_capture["first_epoch"].notna().sum()),
            "max_abs_cos": float(frozen_labels["signed_cos"].abs().max()),
            "upsilon": str(upsilon_tuple(frozen_labels)),
        },
    ]
)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (label, epochs_arr, cos_arr) in zip(
    axes,
    [
        ("free (lens evolves)", traj_epochs, traj_cos),
        ("frozen downstream", frozen_epochs, frozen_cos),
    ],
):
    best_abs_cos = np.abs(cos_arr).max(axis=2)  # (epochs, neurons)
    for neuron_id in range(best_abs_cos.shape[1]):
        ax.plot(epochs_arr, best_abs_cos[:, neuron_id], lw=1.5, label=f"neuron {neuron_id}")
    ax.axhline(COS_THRESHOLD, color="crimson", ls=":", lw=1)
    ax.set_xlabel("epoch")
    ax.set_title(label)
    ax.grid(alpha=0.2)
axes[0].set_ylabel(r"$\max_i |\cos(w_m, u_i)|$")
axes[0].legend(fontsize=8)
fig.suptitle("H11: cutting the feedback loop by freezing the lens", y=1.02)
fig.tight_layout()
plt.show()

## 11. Control B — width-1 first layer (no lens to speak of)

`[1, W, W]`: a single first-layer halfspace with a wide downstream net. The first layer
now has no siblings to co-shape the deep gates, so the construction degenerates toward
a single-index function. Expectation from Track A.3: one weakly aligned direction
rather than a path of distinct ODT nodes.

In [ ]:
DEGENERATE_HIDDEN_DIMS = [1, DEPTH, DEPTH]

degenerate_result = run_relu_cell(SEED, hidden_dims=DEGENERATE_HIDDEN_DIMS)
degenerate_ckpts = degenerate_result["out"]["checkpoint_snapshots"]
degenerate_labels = neuron_labels(
    degenerate_ckpts[max(degenerate_ckpts)], degenerate_result["tree"]
)

print("hidden dims:", DEGENERATE_HIDDEN_DIMS)
print("final metrics:", collect_start_end_metrics(degenerate_result)[-1])
degenerate_labels

## 12. Mini census over seeds (Track A.1)

The DLGN result is about the *set* of reachable solutions, not one run. This is a small
version of the census: fixed data/architecture, varying master seed (which moves both
the dataset and the initialisation), recording the $\upsilon$-tuple and whether it is a
root-to-leaf path solution.

Start small — each seed is a full training run. The full protocol in
`notes/research_spec.md` (Track A.1) is ~200 initialisations x 3-5 dataset resamples
with a data-invariance filter, which belongs in a script under `src/`, not here.

In [ ]:
CENSUS_SEEDS = [5178, 6178, 7178, 8178, 9178]
CENSUS_EPOCHS = TOTAL_EPOCHS

census_rows = []
for seed in CENSUS_SEEDS:
    result = run_relu_cell(
        seed,
        total_epochs=CENSUS_EPOCHS,
        snapshot_epochs=(0, CENSUS_EPOCHS),
        show_progress=False,
    )
    result_ckpts = result["out"]["checkpoint_snapshots"]
    labels = neuron_labels(result_ckpts[max(result_ckpts)], result["tree"])
    metrics = collect_start_end_metrics(result)[-1]
    nodes = sorted(labels.loc[labels["aligned"], "odt_node"])
    leaf = matching_root_to_leaf_path(nodes)
    census_rows.append(
        {
            "seed": seed,
            "upsilon": upsilon_tuple(labels),
            "aligned_neurons": int(labels["aligned"].sum()),
            "distinct_nodes": len(set(nodes)),
            "root_to_leaf_path": leaf is not None,
            "path_leaf": leaf,
            "train_log_loss": metrics["train_log_loss"],
            "test_error": metrics["test_error"],
            "stuck_near_chance": metrics["train_log_loss"] > 0.9 * np.log(2),
        }
    )
    print(f"seed {seed}: {census_rows[-1]['upsilon']}  "
          f"loss={metrics['train_log_loss']:.4f}  err={metrics['test_error']:.4f}")

census_df = pd.DataFrame(census_rows)
census_df

In [ ]:
solution_counts = (
    census_df.assign(upsilon_str=census_df["upsilon"].astype(str))
    .groupby("upsilon_str")
    .agg(
        runs=("seed", "count"),
        root_to_leaf_path=("root_to_leaf_path", "first"),
        mean_test_error=("test_error", "mean"),
    )
    .sort_values("runs", ascending=False)
)
print(f"{census_df['root_to_leaf_path'].sum()}/{len(census_df)} runs "
      "converged to a root-to-leaf path solution")
print(f"{census_df['stuck_near_chance'].sum()}/{len(census_df)} runs stuck near chance")
solution_counts

## 13. What is still missing

Deliberately left out of this notebook, in order of priority
(`notes/research_spec.md`, Track A and B3):

1. **Bespoke initialisation trajectories** (paper Sec. 3.3 analog): start near zero,
   perturb selected neurons by $-\epsilon u_j$, and check that the perturbation steers
   the converged solution the way the hierarchy-seeking narrative predicts. This is the
   sharpest single-path test and needs `model_init` passed to `run_single_relu_seed`.
2. **Second-order probes (H9/H10, CT2):** finite-difference Hessian-vector probes
   across neuron blocks, producing the sign-pattern heatmap to compare against DLGN
   Theorems 1-2.
3. **Full census with the data-invariance filter:** each initialisation re-run on
   several dataset resamples, keeping only runs whose solution is dataset-invariant
   (the paper's 368-of-2000 filter). Belongs in a script, with results under
   `results/`.
4. **`[D, 2, 2]` robustness variant** — is the path solution an artefact of the width-1
   chain?

Any helper here that survives contact with real runs should move into
`src/layer_lenses/relu_analysis.py` (trajectories, labels, capture epochs, lens
profiles) per the repository's notebook-migration policy.